[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-1_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 1: Exact Enumeration for a Small Instance

### Purpose and scope

The vehicle routing problem with backhauls (VRPB) combines two customer types: linehaul customers, who receive goods, and backhaul customers, who provide goods for collection. This notebook finds a minimum-distance route for a small VRPB instance by enumerating customer sequences and checking each sequence against the model's constraints.

The instance has one depot, four delivery customers, three pickup customers, and one vehicle. The input is the customer demand data, vehicle capacity, and a complete distance matrix. The output is the best feasible route or routes found, their distances and load profiles, and constraint checks. The exact enumeration method is suitable for this small example; its factorial growth makes it unsuitable for large instances.

### Model assumptions

- Travel costs are known for every ordered pair of locations. The example matrix is symmetric, although the route-cost calculation uses the directed-arc form and does not require symmetry.
- There is one vehicle, and it starts at and returns to the depot. Every customer is served exactly once on this route.
- The vehicle starts with all goods required for the deliveries. A delivery reduces its load; a pickup increases it.
- All deliveries must be completed before any pickups begin.
- Demands are nonnegative, and the vehicle load must remain between zero and its capacity.
- The model does not include multiple vehicles, time windows, service times, or other operational constraints.

### Why the load and precedence rules matter

The load is the amount of goods physically on the vehicle at each point in the route. It begins at the sum of all delivery demands. After a delivery, the load decreases by that customer's demand; after a pickup, it increases by the pickup demand. Capacity must hold at every point, not just at the start or end. The precedence rule means the route has a delivery phase followed by a pickup phase; a pickup cannot occur while any delivery remains unfinished.

### Mathematical model

Let $D$ be the set of delivery customers, $P$ the set of pickup customers, $N=D\cup P$ the set of all customers, and $n=|N|$. The depot is indexed by $0$. A candidate route is a permutation $\pi=(\pi_1,\ldots,\pi_n)$ of $N$. The exhaustive search considers these permutations and retains those that satisfy precedence and capacity.

For a route $\pi$, total travel distance is

$$
C(\pi)=c_{0,\pi_1}+\sum_{k=1}^{n-1}c_{\pi_k,\pi_{k+1}}+c_{\pi_n,0}.
$$

Here, $c_{ij}$ is the travel cost from location $i$ to location $j$. The first term is travel from the depot to the first customer, the sum is travel between consecutive customers, and the last term is the return to the depot. The function `calculate_route_distance` implements this expression.

Let $d_i$ denote the delivery demand of customer $i\in D$, $p_i$ the pickup demand of customer $i\in P$, and $Q$ the vehicle capacity. The load before the first customer and after each visit is defined recursively by

$$
L_0=\sum_{i\in D}d_i,
\qquad
L_k=
\begin{cases}
L_{k-1}-d_{\pi_k}, & \pi_k\in D,\\
L_{k-1}+p_{\pi_k}, & \pi_k\in P,
\end{cases}
\quad k=1,\ldots,n.
$$

The initial value $L_0$ represents all delivery goods loaded at the depot. The recurrence updates the load after each customer; it is implemented by `calculate_load_profile`. A route is feasible only if it visits every customer exactly once, every delivery precedes every pickup, and

$$
0\leq L_k\leq Q\quad(k=0,\ldots,n).
$$

The optimization problem is to minimize $C(\pi)$ over all feasible permutations. In the code, `check_constraints` tests feasibility, `calculate_route_distance` evaluates the objective, and `solve_vrpb_exhaustive` enumerates the candidate routes. This is an exact permutation-enumeration method; the code does not construct a mixed-integer or network-flow solver model.

With $n$ customers, the search checks $n!$ permutations. Checking a route takes $O(n)$ time, so the search has time complexity $O(n\,n!)$. The implementation also stores feasible routes, which requires memory proportional to the number of feasible permutations retained. The factorial growth is why this approach is intended for small instances.

### Concrete instance

The example contains four deliveries with demands $[5,4,6,3]$, three pickups with demands $[4,5,3]$, vehicle capacity $Q=20$, and an $8\times 8$ symmetric distance matrix including the depot. Customer indices are $1$–$4$ for deliveries and $5$–$7$ for pickups. The stated reference sequence $0\to1\to2\to3\to4\to5\to6\to7\to0$ is feasible and has distance 25 for this matrix. It is a reference route, not a claim that it is optimal; the exhaustive search determines the minimum over all feasible routes.

### How to interpret the results

The search reports the number of permutations checked, the number of feasible routes, the minimum distance, and all routes tied at that minimum within a small numerical tolerance. The load profile includes the load at the depot before departure and the load after each customer visit. A reported route is feasible only when the coverage, precedence, and capacity checks all pass.

In [ ]:
from dataclasses import dataclass
from itertools import permutations
from math import factorial, isclose
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

# A consistent plotting style makes route types and load limits easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for the VRPB example.")

In [ ]:
@dataclass
class VRPBInstance:
    """Store the data that define a single-vehicle VRPB instance."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self):
        self.num_customers = self.num_deliveries + self.num_pickups
        # Customer indices identify demand entries and matrix rows consistently.
        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(
                self.num_deliveries + 1,
                self.num_deliveries + self.num_pickups + 1,
            )
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class Route:
    """Store a route, its load after each visit, and its total distance."""

    sequence: List[int]
    load_profile: List[float]
    distance: float

    def __str__(self):
        route_str = " -> ".join(str(customer) for customer in self.sequence)
        return (
            f"Route: {route_str} | Distance: {self.distance:.2f} | "
            f"Load profile: {self.load_profile}"
        )


print("VRPB data structures are ready.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the specified four-delivery, three-pickup example."""

    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    # Rows and columns use index 0 for the depot, followed by customers 1–7.
    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ]
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(
    f"Created an instance with {instance.num_deliveries} deliveries "
    f"and {instance.num_pickups} pickups."
)
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
def calculate_route_distance(
    route: List[int], distance_matrix: np.ndarray
) -> float:
    """Calculate route distance, including travel to and from the depot."""
    if not route:
        return 0.0

    total_distance = distance_matrix[0, route[0]]

    # Each consecutive arc contributes once to the route's total travel cost.
    for position in range(len(route) - 1):
        total_distance += distance_matrix[route[position], route[position + 1]]

    total_distance += distance_matrix[route[-1], 0]
    return float(total_distance)


def calculate_load_profile(
    route: List[int], instance: VRPBInstance
) -> List[float]:
    """Return the initial vehicle load and the load after each customer visit."""
    current_load = float(sum(instance.delivery_demands))
    load_profile = [current_load]

    for customer in route:
        if customer in instance.delivery_indices:
            demand_index = customer - 1
            current_load -= instance.delivery_demands[demand_index]
        else:
            demand_index = customer - instance.num_deliveries - 1
            current_load += instance.pickup_demands[demand_index]

        load_profile.append(float(current_load))

    return load_profile


def check_constraints(
    route: List[int], instance: VRPBInstance
) -> Tuple[bool, List[str]]:
    """Check customer coverage, delivery precedence, and vehicle capacity."""
    violations = []
    expected_customers = set(instance.all_customer_indices)
    visited_customers = set(route)

    # Set equality alone would not detect a repeated customer, so check route length too.
    if len(route) != len(instance.all_customer_indices) or visited_customers != expected_customers:
        missing = expected_customers - visited_customers
        extra = visited_customers - expected_customers
        if missing:
            violations.append(f"Missing customers: {sorted(missing)}")
        if extra:
            violations.append(f"Unknown customers: {sorted(extra)}")
        if len(route) != len(visited_customers):
            violations.append("A customer is visited more than once.")

    # A pickup can begin only after the final delivery has been completed.
    delivery_positions = [
        position
        for position, customer in enumerate(route)
        if customer in instance.delivery_indices
    ]
    pickup_positions = [
        position
        for position, customer in enumerate(route)
        if customer in instance.pickup_indices
    ]

    if delivery_positions and pickup_positions:
        if min(pickup_positions) < max(delivery_positions):
            violations.append(
                "Precedence constraint violated: a pickup occurs before a delivery."
            )

    # Load limits apply at departure and after every customer visit.
    if not (visited_customers - expected_customers):
        load_profile = calculate_load_profile(route, instance)
        if min(load_profile) < 0:
            violations.append("Capacity constraint violated: vehicle load is below zero.")
        if max(load_profile) > instance.capacity:
            violations.append(
                f"Capacity constraint violated: maximum load {max(load_profile)} "
                f"exceeds capacity {instance.capacity}."
            )

    return len(violations) == 0, violations


print("Route distance, load, and feasibility functions are ready.")

In [ ]:
def solve_vrpb_exhaustive(
    instance: VRPBInstance,
) -> Tuple[List[Route], float, List[List[int]]]:
    """Find every minimum-distance feasible route by permutation enumeration."""
    best_solution = []
    best_distance = float("inf")
    feasible_solutions = []

    all_customers = instance.all_customer_indices
    total_permutations = factorial(instance.num_customers)

    print("Starting exhaustive search for an optimal VRPB route.")
    print(f"Checking permutations of {instance.num_customers} customers...")

    for permutation_index, customer_sequence in enumerate(
        permutations(all_customers), start=1
    ):
        if permutation_index % 10000 == 0:
            print(
                f"  Processed {permutation_index:,} of "
                f"{total_permutations:,} permutations."
            )

        route = list(customer_sequence)
        is_feasible, _ = check_constraints(route, instance)

        if not is_feasible:
            continue

        feasible_solutions.append(route)
        distance = calculate_route_distance(route, instance.distance_matrix)
        load_profile = calculate_load_profile(route, instance)
        route_object = Route(
            sequence=route,
            load_profile=load_profile,
            distance=distance,
        )

        if distance < best_distance and not isclose(
            distance, best_distance, rel_tol=1e-9, abs_tol=1e-9
        ):
            best_distance = distance
            best_solution = [route_object]
        elif isclose(distance, best_distance, rel_tol=1e-9, abs_tol=1e-9):
            best_distance = min(best_distance, distance)
            best_solution.append(route_object)

    print("Exhaustive search completed.")
    print(f"Total permutations checked: {total_permutations:,}")
    print(f"Feasible routes found: {len(feasible_solutions)}")
    if best_solution:
        print(f"Minimum distance: {best_distance}")
        print(f"Routes tied at the minimum within tolerance: {len(best_solution)}")
    else:
        print("No feasible route was found.")

    return best_solution, best_distance, feasible_solutions


print("The exhaustive-search solver is ready.")

In [ ]:
# Enumerating every candidate route makes the minimum exact for this instance.
best_routes, optimal_distance, feasible_solutions = solve_vrpb_exhaustive(instance)

print("\n" + "=" * 60)
print("OPTIMAL ROUTE RESULTS")
print("=" * 60)

if best_routes:
    print(f"\nMinimum total distance: {optimal_distance}")
    print(f"Number of tied optimal routes: {len(best_routes)}")

    for route_number, route in enumerate(best_routes, start=1):
        full_sequence = [0] + route.sequence + [0]
        print(f"\nOptimal route {route_number}:")
        print("Route: " + " -> ".join(map(str, full_sequence)))
        print(f"Distance: {route.distance}")
        print(
            "Load profile (departure, then after each customer): "
            f"{[round(load, 1) for load in route.load_profile]}"
        )

        is_feasible, violations = check_constraints(route.sequence, instance)
        assert is_feasible, f"Reported route violates constraints: {violations}"
        assert isclose(
            calculate_route_distance(route.sequence, instance.distance_matrix),
            route.distance,
            rel_tol=1e-9,
            abs_tol=1e-9,
        )

        delivery_positions = [
            route.sequence.index(customer) for customer in instance.delivery_indices
        ]
        pickup_positions = [
            route.sequence.index(customer) for customer in instance.pickup_indices
        ]
        print(
            "Precedence check: final delivery position "
            f"{max(delivery_positions)}; first pickup position "
            f"{min(pickup_positions)}."
        )
else:
    print("No feasible route is available to report.")

print(f"\nTotal feasible routes: {len(feasible_solutions)}")
if best_routes:
    maximum_load = max(best_routes[0].load_profile)
    print(
        f"Maximum load on an optimal route: {maximum_load} "
        f"of capacity {instance.capacity}."
    )

In [ ]:
def visualize_solution(
    routes: List[Route],
    instance: VRPBInstance,
    title: str = "VRPB Optimal Route",
) -> None:
    """Plot a route, its load profile, the distance matrix, and visit details."""
    if not routes:
        print("No feasible route is available to visualize.")
        return

    route = routes[0]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # Bar colors identify customer type; the legend makes that encoding explicit.
    colors = [
        "tab:red" if customer in instance.delivery_indices else "tab:blue"
        for customer in route.sequence
    ]
    ax1.bar(range(len(route.sequence)), [1] * len(route.sequence), color=colors)
    ax1.set_xlabel("Visit position")
    ax1.set_ylabel("Customer visit")
    ax1.set_title("Customer sequence")
    ax1.set_xticks(range(len(route.sequence)))
    ax1.set_xticklabels(route.sequence)
    ax1.legend(
        handles=[
            Patch(color="tab:red", label="Delivery"),
            Patch(color="tab:blue", label="Pickup"),
        ]
    )
    ax1.grid(axis="y", alpha=0.3)

    load_positions = range(len(route.load_profile))
    ax2.plot(load_positions, route.load_profile, "o-", linewidth=2, markersize=7)
    ax2.axhline(
        y=instance.capacity,
        color="tab:red",
        linestyle="--",
        label=f"Capacity ({instance.capacity})",
    )
    ax2.axhline(y=0, color="black", linewidth=0.8)
    ax2.set_xlabel("Route position (0 is departure)")
    ax2.set_ylabel("Vehicle load (demand units)")
    ax2.set_title("Vehicle load profile")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    sns.heatmap(
        instance.distance_matrix,
        annot=True,
        fmt=".0f",
        cmap="viridis",
        ax=ax3,
        cbar_kws={"label": "Travel cost (problem units)"},
    )
    ax3.set_title("Travel-cost matrix")
    ax3.set_xlabel("Destination location")
    ax3.set_ylabel("Origin location")

    customer_operations = []
    for customer in route.sequence:
        if customer in instance.delivery_indices:
            demand = instance.delivery_demands[customer - 1]
            customer_operations.append(f"Delivery {customer}: -{demand}")
        else:
            demand_index = customer - instance.num_deliveries - 1
            demand = instance.pickup_demands[demand_index]
            customer_operations.append(f"Pickup {customer}: +{demand}")

    ax4.axis("off")
    route_text = " -> ".join(map(str, [0] + route.sequence + [0]))
    details = (
        f"Route:\n{route_text}\n\n"
        f"Total distance: {route.distance}\n\n"
        "Customer operations:\n"
        + "\n".join(customer_operations)
    )
    ax4.text(
        0.05,
        0.95,
        details,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
    )
    ax4.set_title("Route details")

    plt.tight_layout()
    plt.show()


visualize_solution(best_routes, instance)

In [ ]:
def analyze_solution_quality(
    routes: List[Route], instance: VRPBInstance
) -> None:
    """Report precedence, capacity, distance, and load details for a best route."""
    if not routes:
        print("No feasible route is available to analyze.")
        return

    route = routes[0]
    print("DETAILED ROUTE ANALYSIS")
    print("=" * 50)

    delivery_positions = {
        customer: route.sequence.index(customer)
        for customer in instance.delivery_indices
    }
    pickup_positions = {
        customer: route.sequence.index(customer)
        for customer in instance.pickup_indices
    }
    precedence_satisfied = max(delivery_positions.values()) < min(
        pickup_positions.values()
    )

    print("\nPrecedence analysis:")
    print(f"Delivery positions: {delivery_positions}")
    print(f"Pickup positions: {pickup_positions}")
    print(f"Precedence satisfied: {precedence_satisfied}")

    maximum_load = max(route.load_profile)
    minimum_load = min(route.load_profile)
    average_load = float(np.mean(route.load_profile))
    utilization = maximum_load / instance.capacity * 100

    print("\nCapacity analysis:")
    print(f"Vehicle capacity: {instance.capacity}")
    print(f"Maximum load: {maximum_load} ({utilization:.1f}% of capacity)")
    print(f"Minimum load: {minimum_load}")
    print(f"Average load across the recorded profile: {average_load:.2f}")
    print(
        "Capacity constraint satisfied: "
        f"{minimum_load >= 0 and maximum_load <= instance.capacity}"
    )

    depot_to_first = instance.distance_matrix[0, route.sequence[0]]
    customer_to_customer = sum(
        instance.distance_matrix[route.sequence[position], route.sequence[position + 1]]
        for position in range(len(route.sequence) - 1)
    )
    last_to_depot = instance.distance_matrix[route.sequence[-1], 0]
    distance_check = depot_to_first + customer_to_customer + last_to_depot

    print("\nDistance breakdown (problem units):")
    print(f"Depot to first customer: {depot_to_first}")
    print(f"Travel between customers: {customer_to_customer}")
    print(f"Last customer to depot: {last_to_depot}")
    print(f"Total route distance: {route.distance}")
    print(f"Distance recomputation: {distance_check}")
    assert isclose(
        distance_check, route.distance, rel_tol=1e-9, abs_tol=1e-9
    )

    print("\nLoad evolution:")
    print("Position | Customer | Type     | Demand change | Load after visit")
    print("---------|----------|----------|---------------|-----------------")
    for position, customer in enumerate(route.sequence):
        if customer in instance.delivery_indices:
            customer_type = "Delivery"
            demand_change = -instance.delivery_demands[customer - 1]
        else:
            customer_type = "Pickup"
            demand_index = customer - instance.num_deliveries - 1
            demand_change = instance.pickup_demands[demand_index]

        load_after = route.load_profile[position + 1]
        print(
            f"{position + 1:8d} | {customer:8d} | {customer_type:8s} | "
            f"{demand_change:13.1f} | {load_after:16.1f}"
        )

    reference_route = [1, 2, 3, 4, 5, 6, 7]
    reference_distance = calculate_route_distance(
        reference_route, instance.distance_matrix
    )
    reference_feasible, reference_violations = check_constraints(
        reference_route, instance
    )

    print("\nReference-sequence check:")
    print(f"Reference sequence: {reference_route}")
    print(f"Reference distance: {reference_distance}")
    print(f"Reference sequence feasible: {reference_feasible}")
    if reference_violations:
        print(f"Constraint violations: {reference_violations}")
    print(f"Selected optimal sequence: {route.sequence}")
    print(
        "Selected route matches the reference sequence: "
        f"{route.sequence == reference_route}"
    )


analyze_solution_quality(best_routes, instance)

## Summary and conclusions

### What the enumeration establishes

The solver evaluates every customer permutation, checks coverage, delivery-before-pickup precedence, and vehicle capacity, and minimizes the complete depot-to-depot travel distance over the feasible routes. Because every permutation is examined, the minimum found is exact for this instance, subject to the stated model assumptions and numerical comparison tolerance. If several routes have equal minimum distance, the solver reports all routes tied within that tolerance.

For the supplied matrix, the reference sequence $0\to1\to2\to3\to4\to5\to6\to7\to0$ is feasible and has distance 25. The search can also identify a shorter feasible route: for example, $0\to1\to2\to3\to4\to5\to7\to6\to0$ has distance 24. Its load begins at 18, falls to zero after the deliveries, and rises to 12 after the pickups, remaining within capacity 20. These route values follow directly from the supplied demands and distance matrix.

### Strengths and limitations

Exact enumeration is transparent: each candidate is checked against explicit constraints, and the objective is calculated from its arcs. It is useful for understanding the structure of a small VRPB instance and for verifying the feasibility and distance of candidate routes. Its central limitation is factorial growth: with $n$ customers, it checks $n!$ sequences, and the work per sequence grows with route length. Consequently, the method is appropriate for small teaching examples, not large operational instances.

### Interpreting the reported measures

- The route distance includes travel from the depot, between customers, and back to the depot.
- The load profile records departure load and the load after each customer. Its maximum is compared with vehicle capacity, and its minimum is checked against zero.
- Customer coverage and delivery-before-pickup precedence are checked separately from the distance calculation.
- The reported average load is the arithmetic mean of the values in the displayed load profile; it is descriptive and is not part of the optimization objective.

This example illustrates how a VRPB route can be evaluated exactly under a clearly specified model. Additional operational requirements can be represented by additional feasibility checks when the problem scope calls for them.